In [1]:
import pandas as pd

orders = pd.read_csv('C:/Users/Dell\Desktop/ecommerce-churn_prediction/data/raw/olist_orders_dataset.csv', parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])

customers = pd.read_csv('C:/Users/Dell/Desktop/ecommerce-churn_prediction/data/raw/olist_customers_dataset.csv')
reviews = pd.read_csv('C:/Users/Dell/Desktop/ecommerce-churn_prediction/data/raw/olist_order_reviews_dataset.csv')

# Basic shape audit
print("Orders:", orders.shape)
print("Customers:", customers.shape)
print("Reviews:", reviews.shape)

# Null audit
print("\n--- NULL COUNTS ---")
print(orders.isnull().sum())

# Order status distribution
print("\n--- ORDER STATUS ---")
print(orders['order_status'].value_counts())

# Date range of the dataset
print("\n--- DATE RANGE ---")
print(orders['order_purchase_timestamp'].min())
print(orders['order_purchase_timestamp'].max())

# How many unique customers placed more than 1 order?
merged = orders.merge(customers, on='customer_id')
repeat = merged.groupby('customer_unique_id')['order_id'].count()
print("\n--- PURCHASE FREQUENCY ---")
print(repeat.value_counts().head(10))
print(f"\nCustomers with >1 order: {(repeat > 1).sum()} ({(repeat > 1).mean():.1%})")

<>:3: SyntaxWarning: invalid escape sequence '\D'
<>:3: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Dell\AppData\Local\Temp\ipykernel_348\1439251322.py:3: SyntaxWarning: invalid escape sequence '\D'
  orders = pd.read_csv('C:/Users/Dell\Desktop/ecommerce-churn_prediction/data/raw/olist_orders_dataset.csv', parse_dates=[


Orders: (99441, 8)
Customers: (99441, 5)
Reviews: (99224, 7)

--- NULL COUNTS ---
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

--- ORDER STATUS ---
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

--- DATE RANGE ---
2016-09-04 21:15:00
2018-10-17 17:30:00

--- PURCHASE FREQUENCY ---
order_id
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

Customers with >1 order: 2997 (3.1%)


In [2]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────────
import sys
sys.path.append('..')  # so notebook can import from src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'figure.figsize':  (10, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size':       11,
})

print("Environment ready.")

Environment ready.


In [7]:
# # ── Cell 2: Run Pipeline ───────────────────────────────────────────────────────
# ── Cell 2: Run Updated Pipeline ───────────────────────────────────────────────
from importlib import reload
import src.data_pipeline as dp
reload(dp)                          # force reload after code change

from src.data_pipeline import run_pipeline

abt, labels = run_pipeline()        # no arguments anymore

# Sanity check
# print("\nChurn distribution:")
# print(labels['churned_count'].value_counts())
# print(f"\nChurn rate : {labels['churned_count'].mean():.2%}")
# print(f"Class ratio: {labels['churned_count'].sum() / (labels['churned']==0).sum():.1f}:1")
# import src.data_pipeline as dp

# print(dp.__file__)

2026-04-24 16:22:22,790 | INFO | Loading raw data...
2026-04-24 16:22:25,078 | INFO | Orders: (99441, 8) | Customers: (99441, 5) | Reviews: (99224, 7)


DATA QUALITY AUDIT

── ORDERS (99,441 rows × 8 cols) ──
  order_approved_at: 160 nulls (0.2%)
  order_delivered_carrier_date: 1,783 nulls (1.8%)
  order_delivered_customer_date: 2,965 nulls (3.0%)
  Duplicates: 0

── CUSTOMERS (99,441 rows × 5 cols) ──
  Nulls: None
  Duplicates: 0

── REVIEWS (99,224 rows × 7 cols) ──
  review_comment_title: 87,656 nulls (88.3%)
  review_comment_message: 58,247 nulls (58.7%)


2026-04-24 16:22:25,548 | INFO | Cleaned orders: 99,441 → 96,470 (dropped 2,971 non-delivered / null-delivery rows)


  Duplicates: 0

── DATE RANGE ──
  First order: 2016-09-04 21:15:00
  Last order:  2018-10-17 17:30:00

── ORDER STATUS DISTRIBUTION ──
  delivered       96,478  (97.0%)
  shipped          1,107  (1.1%)
  canceled           625  (0.6%)
  unavailable        609  (0.6%)
  invoiced           314  (0.3%)
  processing         301  (0.3%)
  created              5  (0.0%)
  approved             2  (0.0%)

── PURCHASE FREQUENCY DISTRIBUTION ──
order_id
1    93099
2     2745
3      203
4       30
5        8
6        6
7        3
9        1

  Customers with >1 order: 2,997 (3.1%)


2026-04-24 16:22:25,549 | INFO | Building Analytical Base Table (ABT)...
2026-04-24 16:22:25,830 | INFO | ABT shape: (96470, 16)
2026-04-24 16:22:25,831 | INFO | Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'review_score', 'review_creation_date', 'delivery_delay_days', 'approval_delay_hours']
2026-04-24 16:22:25,831 | INFO | Engineering churn labels (v2 — repeat purchase approach)...
2026-04-24 16:22:26,306 | INFO | Total unique customers  : 93,350
2026-04-24 16:22:26,306 | INFO | Will return  (1, minority) : 2,801  (3.0%)
2026-04-24 16:22:26,307 | INFO | One-time     (0, majority) : 90,549  (97.0%)
2026-04-24 16:22:26,307 | INFO | Class ratio  : 32.3:1  (majority:minority)
2026-04-24 16:22:27,259 | INFO | Saved → C:\Users\Dell\Desktop\ecommerce-chur

In [11]:
# ── Cell 3: Sanity Check Outputs ───────────────────────────────────────────────
print("ABT shape:    ", abt.shape)
print("Labels shape: ", labels.shape)
print("\n── ABT sample ──")
display(abt.head(3))
print("\n── Labels sample ──")
display(labels.head(3))
print("\n── Churn distribution ──")
#print(labels['churned'].value_counts())
#print(f"\nChurn rate: {labels['churned'].mean():.2%}")

ABT shape:     (96470, 16)
Labels shape:  (93350, 18)

── ABT sample ──


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,review_score,review_creation_date,delivery_delay_days,approval_delay_hours
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:00,2017-10-02 11:07:00,2017-10-04 19:55:00,2017-10-10 21:25:00,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,4.0,2017-10-11,-8,0.18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:00,2018-07-26 03:24:00,2018-07-26 14:31:00,2018-08-07 15:27:00,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,4.0,2018-08-08,-6,30.72
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:00,2018-08-08 08:55:00,2018-08-08 13:50:00,2018-08-17 18:06:00,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,5.0,2018-08-18,-18,0.28



── Labels sample ──


,customer_unique_id,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_zip_code_prefix,customer_city,customer_state,review_score,review_creation_date,delivery_delay_days,approval_delay_hours,total_orders,will_return
0,0000366f3b9a7992bf8c76cfdf3221e2,e22acc9c116caa3f2b7121bbb380d08e,fadbb3709178fc513abc1b2670aa1ad2,delivered,2018-05-10 10:56:00,2018-05-10 11:11:00,2018-05-12 08:18:00,2018-05-16 20:48:00,2018-05-21,7787,cajamar,SP,5.0,2018-05-17,-5,0.25,1,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,3594e05a005ac4d06a72673270ef9ec9,4cb282e167ae9234755102258dd52ee8,delivered,2018-05-07 11:11:00,2018-05-07 18:25:00,2018-05-09 12:18:00,2018-05-10 18:02:00,2018-05-15,6053,osasco,SP,4.0,2018-05-11,-5,7.23,1,0
2,0000f46a3911fa3c0805444483337064,b33ec3b699337181488304f362a6b734,9b3932a6253894a02c1df9d19004239f,delivered,2017-03-10 21:05:00,2017-03-10 21:05:00,2017-03-13 12:58:00,2017-04-05 14:38:00,2017-04-07,88115,sao jose,SC,3.0,2017-04-06,-2,0.00,1,0



── Churn distribution ──


In [9]:
# ── Cell 4: INSIGHT 1 — Churn Label Distribution ──────────────────────────────
# fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# # Count plot
# # Always index explicitly, never rely on value_counts() order
# churned_count  = labels['churned'].sum()          # 61876
# retained_count = (labels['churned'] == 0).sum()   # 421
# counts = labels['churned'].value_counts().sort_index()


# axes[0].bar(
#     ['Retained (0)', 'Churned (1)'],
#     counts.values,
#     color=['#2ecc71', '#e74c3c']
# )

# for i, v in enumerate(counts.values):
#     axes[0].text(i, v + 100, f'{v:,}\n({v/len(labels):.1%})', ha='center')

# # Pie chart
# axes[1].pie(
#     counts.values,
#     labels=['Retained', 'Churned'],
#     colors=["#2e43cc", '#e74c3c'],
#     autopct='%1.1f%%'
# )

# plt.suptitle('INSIGHT 1: Class Imbalance Overview', fontsize=14, fontweight='bold', y=1.02)
# plt.tight_layout()
# plt.savefig('../data/processed/insight1_churn_distribution.png', bbox_inches='tight', dpi=150)
# plt.show()

# print("""
# KEY TAKEAWAY:
# Your data will be heavily imbalanced (most customers churn in a marketplace).
# You CANNOT use accuracy as a metric — a model predicting everyone churns gets ~95% accuracy.
# You MUST use: AUC-ROC, Precision-Recall AUC, F1 on minority class.
# """)